# Molecular docking -- Mentha_spicata vs Aldose Reductase (2FZD)

Protein: Human aldose reductase, PDB 2FZD, native inhibitor tolrestat (auto-detected in Section 1, chain A only -- this structure's own publication notes some tolrestat-AR crystal forms picked up extra, non-biological copies from crystal packing). The required NADP+ cofactor (ligand code NAP) is present in the crystal but excluded from both the native-ligand selection and the receptor PDB, since rigid-receptor Vina docking does not model cofactor-dependent chemistry regardless -- a general limitation, not specific to this pipeline. Compounds: Carvone, Limonene, cis-Dihydrocarvone, beta-Caryophyllene (Bardaweel et al. 2018).

## Content of this notebook

1. Feching system and cleanup
2. Protein and ligand sanitization
3. System Visualization 
4. Docking with AutoDock Vina
    - Receptor preparation
    - Ligand preparation
    - Docking box definition
    - Docking
    - PDBQT results file conversion to SDF
    - 3D visualization of docking results
    - 2D interaction table and map
5. Docking with Smina
    - Receptor preparation
    - Ligand preparation
    - Docking box definition
    - Docking
    - 3D visualization of docking results
    - 2D interaction table and map
6. Docking with LeDock
    - Receptor preparation
    - Ligand preparation
    - Docking box definition
    - Docking
    - DOK results file conversion to SDF
    - 3D visualization of docking results
    - 2D interaction table and map

In [ ]:
from pymol import cmd
import py3Dmol

from vina import Vina

from openbabel import pybel

from rdkit import Chem
from rdkit.Chem import AllChem, Draw

from meeko import MoleculePreparation

import MDAnalysis as mda
from MDAnalysis.coordinates import PDB

import prolif as plf
from prolif.plotting.network import LigNetwork


import sys, os
sys.path.insert(1, "/home/ali/medical plants/docking/Jupyter_Dock-main/utilities/")
from utils import fix_protein, getbox, generate_ledock_file, pdbqt_to_sdf, dok_to_sdf


import warnings
warnings.filterwarnings("ignore")
%config Completer.use_jedi = False

It's a good idea to run the test protocols before attempting custom projects. Later, the user can specify the location of his/her project and save all of the files separately.

In [ ]:
os.chdir('/home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/')


## 1. Feching system and cleanup

Implementing Pymol is a simple way to download PDB structures. The user can launch this or any other Jupyter Dock's protocol by providing his or her own files.

In [ ]:
cmd.fetch(code='2FZD',type='pdb1')
cmd.select(name='Prot',selection='polymer.protein')
cmd.save(filename='2FZD_clean.pdb',format='pdb',selection='Prot',state=1)

# 2FZD's real inhibitor is tolrestat. Auto-detect rather than hardcoding a
# guessed code: exclude the required NADP+ cofactor (NAP) and water, and
# restrict to chain A -- this structure's own publication notes that some
# tolrestat-aldose-reductase crystal forms picked up EXTRA, non-biological
# copies from crystal packing effects (up to 4 in one related structure),
# so we deliberately avoid grabbing more than one copy.
cmd.select(name='NativeLig', selection='organic and not resn NAP+NDP+HOH and chain A')
_natlig_model = cmd.get_model('NativeLig', state=1)
_natlig_resns = sorted(set((a.resn, a.resi) for a in _natlig_model.atom))
print("Native ligand candidate(s) found (resn, resi):", _natlig_resns)
print("Total atom count:", len(_natlig_model.atom))

# If more than one distinct (resn, resi) pair appears above, narrow the
# selection to a single one, e.g.:
# cmd.select(name='NativeLig', selection="resn XXX and resi YYY and chain A")
cmd.save(filename='2FZD_native_lig.mol2', format='mol2', selection='NativeLig', state=1)
cmd.delete('all')


## 2. Protein and ligand sanitization

### 2.1. Protein Sanitization

#### Method 1: LePro

The Lephar molecular docking suite includes a very powerful tool for automatically preparing proteins for molecular docking. As a result, for these protocols, it will be the preferred tool for protein preparation. The protein prepared by LePro can be used in AutoDock VIna and LeDock.

In [ ]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/lepro_linux_x86" # Launch this cell to see parameters


In [ ]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/lepro_linux_x86" {'2FZD_clean.pdb'}

os.rename('pro.pdb','2FZD_clean_H.pdb')


#### Method 2: fix_protein (PDBFixer)

For proteins with missing amino acids or residues, or to ensure a more thorough sanitization of protein Jupyter Dock includes the _fix_protein()_ function, which employs PDBFixer to correct a wide range of common errors in protein pdb files. Furthermore, PDBFixer enables the assignment of pH-dependent protonation states to proteins.


>**Warning 1:**  It is possible to encounter problems with protein fixing when using **_fix_protein()_** and AutoDock Tools' **_prepare_receptor_** or when running LeDock. To resolve the issue, it is best to set the parameter _-A hydrogens_ in _prepare receptor_.


>**Hint:** PDBFixer is a great solution for many systems because it can solve serious problems in PDB files. As a result, PDBFixer renumbers the residues beginning with 1 regardless of the numbering on the original PDB file. To address this issue, the **_fix_protein()_** function includes a protocol for atomically renumbering the residues in accordance with the original PDB file.


**_fix_protein ( params )_**

Params:
      
  - **filename**: _str or path-like_ ; input file containing protein struture to be modified, file extrension must be pdb

  - **addHs_pH**: _float_ ; Add hydrogens at user defined pH

  - **try_renumberResidues**: _bool_ ; By default PDBFixer renumarets residues starting in 1, this option tries to recover originar residues numbering
    
  - **output**: _str or path-like_ ; output filename, extension must be pdb

```
fix_protein(filename='2FZD_clean.pdb',addHs_pH=7.4,try_renumberResidues=True,output='2FZD_clean_H.pdb')
```

## 4. Docking with AutoDock Vina

AutoDock Vina (Vina) is one of the docking engines in the AutoDock Suite, together with AutoDock4 (AD4), AutoDockGPU, AutoDockFR, and AutoDock-CrankPep, and arguably among the most widely used and successful docking engines. The reasons for this success are mostly due to its ease of use and its speed (up to 100x faster than AD4), when compared to the other docking engines in the suite and elsewhere, as well as being open source.

### 4.1. Protein preparation

After sanitization, the protein docking preparation includes converting it to the PDBQT file format, which stores the atomic coordinates, partial charges, and AutoDock atom types for both the receptor and the ligand. 

>**Hint:** Despite the fact that the PDBQT format includes changes (Q) and atom types (T) for molecular docking. The charges are not required for Autodock Vina, which computes electrostatic interactions using its own force field. Although, when using AutoDock instead of AutoDock Vina, the Q term is required. More information can be found here:https://autodock-vina.readthedocs.io/en/latest/introduction.html

#### Method 1: AutoDock Tools prepare_receptor

The AutoDock Tools are the best way to prepare the receptor for AutoDock Vina. Nonetheless, AutoDock Tools is a comprehensive suite of programs and scripts that are difficult to manage. Jupyter Dock execute the **_prepare receptor_** and **_prepare_ligand_** functions on their own. As a result, obtaining proper PDBQT files for AutoDock Vina has never been easier.

>**Warning:** There is currently only one method for preparing a PDBQT file for a receptor. It is expected that new methods apart from AutoDock Tools executables will be included in the near future into Jupyter Dock.

In [ ]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_receptor" #Lauch this cell to see parameters


In [ ]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_receptor" -v -r {'2FZD_clean_H.pdb'} -o {'2FZD_clean_H.pdbqt'}

# If this raises "UnboundLocalError: local variable 'bonds' referenced before
# assignment", use this fallback:
# import subprocess
# r = subprocess.run(["obabel", "2FZD_clean_H.pdb", "-O", "2FZD_clean_H.pdbqt", "-xr"],
#                     capture_output=True, text=True)
# print(r.stdout, r.stderr)
#
# Note: prepare_receptor may also warn about the NADP+ cofactor if any part
# of it remained in the file (it shouldn't, since only 'Prot' -- polymer
# atoms -- was saved to 2FZD_clean.pdb above). If AR's activity genuinely
# requires NADP+ to be present for realistic docking (it's a cofactor-
# dependent oxidoreductase), that's a modeling limitation of rigid-receptor
# small-molecule docking generally, not specific to this pipeline.


In [ ]:
fix_protein(filename='2FZD_clean.pdb',addHs_pH=7.4,try_renumberResidues=True,output='2FZD_clean_H_fix.pdb')

## 7. Screening PHYTO compounds (Mentha_spicata) against Aldose Reductase (2FZD)

Reuses the receptor (`2FZD_clean_H.pdb`/`.pdbqt`) and the docking box
(native ligand tolrestat, chain A, `state=1`) prepared above. Each
compound docked in its own cell with its own uniquely-named variables,
seed pinned at 42. Source: Bardaweel et al. 2018 (Laghouat). SMILES
verified against PubChem/RDKit formula cross-check (Carvone and Limonene
both corrected in this project after an earlier SMILES error).

**Direct literature test this round**: Limonene is one of these four
compounds, and D-limonene's published AR docking score (-6.7 kcal/mol,
essential-oil-vs-AR study) is the exact reference value the previous three
Lavandula rounds have all clustered around (-5.8 to -6.8 kcal/mol). This
is the first chance in this project to compare the SAME compound directly,
rather than a structural analog.

In [ ]:
import os
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors
from openbabel import openbabel as ob
from vina import Vina
import pandas as pd
import matplotlib.pyplot as plt

DOCKING_DIR = "/home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/"
FIGURES_DIR = os.path.join(DOCKING_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.chdir(DOCKING_DIR)

RECEPTOR_PDB   = os.path.join(DOCKING_DIR, "2FZD_clean_H.pdb")
RECEPTOR_PDBQT = os.path.join(DOCKING_DIR, "2FZD_clean_H.pdbqt")
NATIVE_LIG     = os.path.join(DOCKING_DIR, "2FZD_native_lig.mol2")

assert os.path.exists(RECEPTOR_PDBQT), "Run the receptor prep cells above first."
assert os.path.exists(NATIVE_LIG), "Run the native ligand extraction cell above first."

VINA_SEED = 42  # pinned for reproducibility

def prepare_ligand_from_smiles(name, smiles, n_confs=20):
    """SMILES -> multiple 3D conformers (ETKDGv3) -> MMFF94 minimize each ->
    keep lowest-energy conformer -> PDB -> PDBQT (AutoDockTools prepare_ligand,
    with OpenBabel fallback)."""
    import subprocess
    m = Chem.MolFromSmiles(smiles)
    m = Chem.AddHs(m)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    conf_ids = AllChem.EmbedMultipleConfs(m, numConfs=n_confs, params=params)

    energies = []
    for cid in conf_ids:
        ff = AllChem.MMFFGetMoleculeForceField(m, AllChem.MMFFGetMoleculeProperties(m), confId=cid)
        if ff is None:
            continue
        ff.Minimize(maxIts=2000)
        energies.append((ff.CalcEnergy(), cid))
    if not energies:
        for cid in conf_ids:
            ff = AllChem.UFFGetMoleculeForceField(m, confId=cid)
            ff.Minimize(maxIts=2000)
            energies.append((ff.CalcEnergy(), cid))

    best_energy, best_cid = min(energies, key=lambda x: x[0])
    print(f"  {name}: {len(conf_ids)} conformers generated, "
          f"lowest energy = {best_energy:.2f} kcal/mol (conf {best_cid})")

    pdb_path   = os.path.join(DOCKING_DIR, f'_lig_{name}.pdb')
    pdbqt_path = os.path.join(DOCKING_DIR, f'_lig_{name}.pdbqt')
    Chem.MolToPDBFile(m, pdb_path, confId=best_cid)

    PREP_LIG_BIN = "/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_ligand"
    result = subprocess.run([PREP_LIG_BIN, "-v", "-l", pdb_path, "-o", pdbqt_path],
                             capture_output=True, text=True)
    if not os.path.exists(pdbqt_path) or os.path.getsize(pdbqt_path) == 0:
        obConv = ob.OBConversion()
        obConv.SetInAndOutFormats('pdb', 'pdbqt')
        obmol = ob.OBMol()
        obConv.ReadFile(obmol, pdb_path)
        obConv.WriteFile(obmol, pdbqt_path)

    return pdbqt_path


### 7.1. Recompute the docking box (native ligand tolrestat, `state=1`)

In [ ]:
cmd.load(filename=RECEPTOR_PDB, format='pdb', object='prot', state=1)
cmd.load(filename=NATIVE_LIG, format='mol2', object='natlig', state=1)

center, size = getbox(selection='natlig', extending=5.0, software='vina')
cmd.delete('all')

print(center, '\n', size)


### 7.2. Dock **Carvone** against Aldose Reductase (2FZD)

In [ ]:
# ---- Carvone (Mentha_spicata, Bardaweel et al. 2018) ----
smiles_Carvone = "CC1=CCC(CC1=O)C(=C)C"
lig_pdbqt_Carvone = prepare_ligand_from_smiles("Carvone", smiles_Carvone)

v_Carvone = Vina(sf_name='vina', seed=VINA_SEED)
v_Carvone.set_receptor(RECEPTOR_PDBQT)
v_Carvone.set_ligand_from_file(lig_pdbqt_Carvone)
v_Carvone.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                     box_size=[size['size_x'], size['size_y'], size['size_z']])

v_Carvone.dock(exhaustiveness=10, n_poses=10)
vina_out_pdbqt_Carvone = os.path.join(DOCKING_DIR, "Carvone_2FZD_vina_out.pdbqt")
v_Carvone.write_poses(vina_out_pdbqt_Carvone, n_poses=10, overwrite=True)

energies_Carvone = v_Carvone.energies(n_poses=10)
best_score_Carvone = round(float(energies_Carvone[0][0]), 3)

print("=== Carvone ===")
print(f"Best Vina score: {best_score_Carvone} kcal/mol")
for i, e in enumerate(energies_Carvone):
    print(f"  Pose {i+1}: {e[0]:.3f}")

vina_out_sdf_Carvone = vina_out_pdbqt_Carvone.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_Carvone, output=vina_out_sdf_Carvone)


### 7.3. Dock **Limonene** against Aldose Reductase (2FZD)

In [ ]:
# ---- Limonene (Mentha_spicata, Bardaweel et al. 2018) ----
smiles_Limonene = "CC1=CCC(CC1)C(=C)C"
lig_pdbqt_Limonene = prepare_ligand_from_smiles("Limonene", smiles_Limonene)

v_Limonene = Vina(sf_name='vina', seed=VINA_SEED)
v_Limonene.set_receptor(RECEPTOR_PDBQT)
v_Limonene.set_ligand_from_file(lig_pdbqt_Limonene)
v_Limonene.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                     box_size=[size['size_x'], size['size_y'], size['size_z']])

v_Limonene.dock(exhaustiveness=10, n_poses=10)
vina_out_pdbqt_Limonene = os.path.join(DOCKING_DIR, "Limonene_2FZD_vina_out.pdbqt")
v_Limonene.write_poses(vina_out_pdbqt_Limonene, n_poses=10, overwrite=True)

energies_Limonene = v_Limonene.energies(n_poses=10)
best_score_Limonene = round(float(energies_Limonene[0][0]), 3)

print("=== Limonene ===")
print(f"Best Vina score: {best_score_Limonene} kcal/mol")
for i, e in enumerate(energies_Limonene):
    print(f"  Pose {i+1}: {e[0]:.3f}")

vina_out_sdf_Limonene = vina_out_pdbqt_Limonene.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_Limonene, output=vina_out_sdf_Limonene)


### 7.4. Dock **cis_Dihydrocarvone** against Aldose Reductase (2FZD)

In [ ]:
# ---- cis_Dihydrocarvone (Mentha_spicata, Bardaweel et al. 2018) ----
smiles_cis_Dihydrocarvone = "C[C@@H]1CC[C@@H](CC1=O)C(=C)C"
lig_pdbqt_cis_Dihydrocarvone = prepare_ligand_from_smiles("cis_Dihydrocarvone", smiles_cis_Dihydrocarvone)

v_cis_Dihydrocarvone = Vina(sf_name='vina', seed=VINA_SEED)
v_cis_Dihydrocarvone.set_receptor(RECEPTOR_PDBQT)
v_cis_Dihydrocarvone.set_ligand_from_file(lig_pdbqt_cis_Dihydrocarvone)
v_cis_Dihydrocarvone.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                     box_size=[size['size_x'], size['size_y'], size['size_z']])

v_cis_Dihydrocarvone.dock(exhaustiveness=10, n_poses=10)
vina_out_pdbqt_cis_Dihydrocarvone = os.path.join(DOCKING_DIR, "cis_Dihydrocarvone_2FZD_vina_out.pdbqt")
v_cis_Dihydrocarvone.write_poses(vina_out_pdbqt_cis_Dihydrocarvone, n_poses=10, overwrite=True)

energies_cis_Dihydrocarvone = v_cis_Dihydrocarvone.energies(n_poses=10)
best_score_cis_Dihydrocarvone = round(float(energies_cis_Dihydrocarvone[0][0]), 3)

print("=== cis_Dihydrocarvone ===")
print(f"Best Vina score: {best_score_cis_Dihydrocarvone} kcal/mol")
for i, e in enumerate(energies_cis_Dihydrocarvone):
    print(f"  Pose {i+1}: {e[0]:.3f}")

vina_out_sdf_cis_Dihydrocarvone = vina_out_pdbqt_cis_Dihydrocarvone.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_cis_Dihydrocarvone, output=vina_out_sdf_cis_Dihydrocarvone)


### 7.5. Dock **beta_Caryophyllene** against Aldose Reductase (2FZD)

In [ ]:
# ---- beta_Caryophyllene (Mentha_spicata, Bardaweel et al. 2018) ----
smiles_beta_Caryophyllene = "CC1=CCCC(=C)C2CC(C2CC1)(C)C"
lig_pdbqt_beta_Caryophyllene = prepare_ligand_from_smiles("beta_Caryophyllene", smiles_beta_Caryophyllene)

v_beta_Caryophyllene = Vina(sf_name='vina', seed=VINA_SEED)
v_beta_Caryophyllene.set_receptor(RECEPTOR_PDBQT)
v_beta_Caryophyllene.set_ligand_from_file(lig_pdbqt_beta_Caryophyllene)
v_beta_Caryophyllene.compute_vina_maps(center=[center['center_x'], center['center_y'], center['center_z']],
                     box_size=[size['size_x'], size['size_y'], size['size_z']])

v_beta_Caryophyllene.dock(exhaustiveness=10, n_poses=10)
vina_out_pdbqt_beta_Caryophyllene = os.path.join(DOCKING_DIR, "beta_Caryophyllene_2FZD_vina_out.pdbqt")
v_beta_Caryophyllene.write_poses(vina_out_pdbqt_beta_Caryophyllene, n_poses=10, overwrite=True)

energies_beta_Caryophyllene = v_beta_Caryophyllene.energies(n_poses=10)
best_score_beta_Caryophyllene = round(float(energies_beta_Caryophyllene[0][0]), 3)

print("=== beta_Caryophyllene ===")
print(f"Best Vina score: {best_score_beta_Caryophyllene} kcal/mol")
for i, e in enumerate(energies_beta_Caryophyllene):
    print(f"  Pose {i+1}: {e[0]:.3f}")

vina_out_sdf_beta_Caryophyllene = vina_out_pdbqt_beta_Caryophyllene.replace('.pdbqt', '.sdf')
pdbqt_to_sdf(pdbqt_file=vina_out_pdbqt_beta_Caryophyllene, output=vina_out_sdf_beta_Caryophyllene)


### 7.6. Total docking results table -- Mentha_spicata vs Aldose Reductase (2FZD)

In [ ]:
docking_rows_spicata_AR = [
    {"Compound": "Carvone",            "Target": "Aldose_reductase (2FZD)", "Best_Vina_Score_kcal_mol": best_score_Carvone},
    {"Compound": "Limonene",           "Target": "Aldose_reductase (2FZD)", "Best_Vina_Score_kcal_mol": best_score_Limonene},
    {"Compound": "cis_Dihydrocarvone", "Target": "Aldose_reductase (2FZD)", "Best_Vina_Score_kcal_mol": best_score_cis_Dihydrocarvone},
    {"Compound": "beta_Caryophyllene", "Target": "Aldose_reductase (2FZD)", "Best_Vina_Score_kcal_mol": best_score_beta_Caryophyllene},
]

docking_df_spicata_AR = pd.DataFrame(docking_rows_spicata_AR).sort_values("Best_Vina_Score_kcal_mol").reset_index(drop=True)
docking_df_spicata_AR.to_csv(os.path.join(DOCKING_DIR, "mentha_spicata_vs_AldoseReductase_vina.csv"), index=False)
docking_df_spicata_AR


### 7.7. Figure -- docking score comparison, Mentha_spicata vs Aldose Reductase

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#2d6a4f' if s == docking_df_spicata_AR['Best_Vina_Score_kcal_mol'].min() else '#74c69d'
          for s in docking_df_spicata_AR['Best_Vina_Score_kcal_mol']]
ax.barh(docking_df_spicata_AR['Compound'], docking_df_spicata_AR['Best_Vina_Score_kcal_mol'], color=colors)
ax.set_xlabel('Best Vina docking score (kcal/mol)')
ax.set_title('Mentha spicata compounds vs. Aldose Reductase (2FZD)')
ax.invert_yaxis()
for i, v in enumerate(docking_df_spicata_AR['Best_Vina_Score_kcal_mol']):
    ax.text(v - 0.15, i, f"{v:.2f}", va='center', ha='right', fontsize=9, color='white')

# Reference line: D-limonene's published AR docking score (-6.7 kcal/mol)
ax.axvline(-6.7, color='red', linestyle='--', linewidth=1, alpha=0.7, label='D-limonene (literature, -6.7)')
ax.legend(loc='lower right', fontsize=8)

plt.tight_layout()

fig_path_spicata_AR = os.path.join(FIGURES_DIR, 'mentha_spicata_vs_AldoseReductase_scores.png')
plt.savefig(fig_path_spicata_AR, dpi=300, bbox_inches='tight')
plt.show()
print("Saved figure:", fig_path_spicata_AR)


### 7.8. ProLif interaction fingerprints -- Mentha_spicata vs Aldose Reductase (each compound, own cell)

`n_jobs=1` avoids a multiprocessing/pickling crash in this environment;
`to_dataframe()` takes no arguments in ProLif 2.2.1.

In [ ]:
prot_fp = mda.Universe("2FZD_clean_H_fix.pdb")  # fix_protein() output
prot_fp = plf.Molecule.from_mda(prot_fp)


In [ ]:
# ---- ProLif: Carvone ----
lig_suppl_Carvone = list(plf.sdf_supplier(vina_out_sdf_Carvone))
fp_Carvone = plf.Fingerprint()
fp_Carvone.run_from_iterable(lig_suppl_Carvone, prot_fp, n_jobs=1)
results_df_Carvone = fp_Carvone.to_dataframe()

print("=== Carvone ===")
display(results_df_Carvone)

net_Carvone = fp_Carvone.plot_lignetwork(lig_suppl_Carvone[0], kind="frame", frame=0, rotation=270)
net_png_Carvone = os.path.join(FIGURES_DIR, "Carvone_2FZD_2Dnetwork.png")
net_Carvone.save_png(net_png_Carvone)
print("Saved network figure:", net_png_Carvone)


In [ ]:
# ---- ProLif: Limonene ----
lig_suppl_Limonene = list(plf.sdf_supplier(vina_out_sdf_Limonene))
fp_Limonene = plf.Fingerprint()
fp_Limonene.run_from_iterable(lig_suppl_Limonene, prot_fp, n_jobs=1)
results_df_Limonene = fp_Limonene.to_dataframe()

print("=== Limonene ===")
display(results_df_Limonene)

net_Limonene = fp_Limonene.plot_lignetwork(lig_suppl_Limonene[0], kind="frame", frame=0, rotation=270)
net_png_Limonene = os.path.join(FIGURES_DIR, "Limonene_2FZD_2Dnetwork.png")
net_Limonene.save_png(net_png_Limonene)
print("Saved network figure:", net_png_Limonene)


In [ ]:
# ---- ProLif: cis_Dihydrocarvone ----
lig_suppl_cis_Dihydrocarvone = list(plf.sdf_supplier(vina_out_sdf_cis_Dihydrocarvone))
fp_cis_Dihydrocarvone = plf.Fingerprint()
fp_cis_Dihydrocarvone.run_from_iterable(lig_suppl_cis_Dihydrocarvone, prot_fp, n_jobs=1)
results_df_cis_Dihydrocarvone = fp_cis_Dihydrocarvone.to_dataframe()

print("=== cis_Dihydrocarvone ===")
display(results_df_cis_Dihydrocarvone)

net_cis_Dihydrocarvone = fp_cis_Dihydrocarvone.plot_lignetwork(lig_suppl_cis_Dihydrocarvone[0], kind="frame", frame=0, rotation=270)
net_png_cis_Dihydrocarvone = os.path.join(FIGURES_DIR, "cis_Dihydrocarvone_2FZD_2Dnetwork.png")
net_cis_Dihydrocarvone.save_png(net_png_cis_Dihydrocarvone)
print("Saved network figure:", net_png_cis_Dihydrocarvone)


In [ ]:
# ---- ProLif: beta_Caryophyllene ----
lig_suppl_beta_Caryophyllene = list(plf.sdf_supplier(vina_out_sdf_beta_Caryophyllene))
fp_beta_Caryophyllene = plf.Fingerprint()
fp_beta_Caryophyllene.run_from_iterable(lig_suppl_beta_Caryophyllene, prot_fp, n_jobs=1)
results_df_beta_Caryophyllene = fp_beta_Caryophyllene.to_dataframe()

print("=== beta_Caryophyllene ===")
display(results_df_beta_Caryophyllene)

net_beta_Caryophyllene = fp_beta_Caryophyllene.plot_lignetwork(lig_suppl_beta_Caryophyllene[0], kind="frame", frame=0, rotation=270)
net_png_beta_Caryophyllene = os.path.join(FIGURES_DIR, "beta_Caryophyllene_2FZD_2Dnetwork.png")
net_beta_Caryophyllene.save_png(net_png_beta_Caryophyllene)
print("Saved network figure:", net_png_beta_Caryophyllene)


### 7.9. Figure -- best-scoring compound vs. the real native ligand (tolrestat)

In [ ]:
best_row_spicata_AR = docking_df_spicata_AR.iloc[0]
best_compound_spicata_AR = best_row_spicata_AR['Compound']
print("Best-scoring compound:", best_compound_spicata_AR, "|", best_row_spicata_AR['Best_Vina_Score_kcal_mol'], "kcal/mol")

best_sdf_spicata_AR = {
    "Carvone": vina_out_sdf_Carvone,
    "Limonene": vina_out_sdf_Limonene,
    "cis_Dihydrocarvone": vina_out_sdf_cis_Dihydrocarvone,
    "beta_Caryophyllene": vina_out_sdf_beta_Caryophyllene,
}[best_compound_spicata_AR]

view = py3Dmol.view()
view.removeAllModels()
view.setViewStyle({'style':'outline','color':'black','width':0.1})

view.addModel(open(RECEPTOR_PDB,'r').read(), format='pdb')
Prot = view.getModel()
Prot.setStyle({'cartoon':{'arrows':True, 'tubes':True, 'style':'oval', 'color':'white'}})
view.addSurface(py3Dmol.VDW, {'opacity':0.6,'color':'white'})

view.addModel(open(NATIVE_LIG,'r').read(), format='mol2')
ref_m = view.getModel()
ref_m.setStyle({}, {'stick':{'colorscheme':'magentaCarbon','radius':0.2}})

results_spi_AR = Chem.SDMolSupplier(best_sdf_spicata_AR)
p_spi_AR = Chem.MolToMolBlock(results_spi_AR[0], False)
print('Reference (tolrestat): Magenta | Best compound pose: Cyan')
print('Pose: {} | Score: {}'.format(results_spi_AR[0].GetProp('Pose'), results_spi_AR[0].GetProp('Score')))

view.addModel(p_spi_AR, 'mol')
x = view.getModel()
x.setStyle({}, {'stick':{'colorscheme':'cyanCarbon','radius':0.2}})

view.zoomTo()
view.show()
